# 학습/검증 파일이 선언과 같은지 확인하기

## 이번 질문

이 노트북은 시간이 남을 때 여는 선택 실습입니다. 본편에서 데이터 역할을 이미 구분한 뒤에만 실행합니다.

지금 폴더의 학습 파일과 검증 파일이, 과정이 선언한 분할과 같은 내용인지 직접 계산합니다. 공식 평가 파일과 정답 없는 운영 파일은 열지 않습니다. 지문이 같다고 해서 모델을 승인했거나 배포가 끝난 것은 아닙니다.

## 먼저 예상

학습 파일의 행 수와 내용 지문이 선언 파일과 같을지 한 문장으로 적습니다. 데이터 잠금 파일 전체 지문까지 지금 폴더와 반드시 같아야 하는지도 예상합니다.

## 실행과 관측

### 1. 선언과 역할 표 읽기

이 노트북은 저장소 루트를 찾아 선언 파일과 역할 표를 엽니다. `labs/ch01-data-quality/`에서 커널을 시작해도 같은 경로를 사용합니다. 역할별 행 수는 분할 표에서 읽고, 파일 내용은 학습/검증만 지문 계산에 사용합니다. 분할 표나 학습/검증 파일이 없으면 `uv run python scripts/setup_course.py --data-only`로 실습 데이터를 준비합니다.

In [ ]:
import json
from pathlib import Path

import pandas as pd

ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir()
)
# 과정이 선언한 분할 정보입니다. 이 파일을 고치지 않습니다.
lineage_path = ROOT / "docs/reference/evidence/data-lineage/split-revision-v2.json"
# 지금 폴더의 역할 표입니다. 한 행이 한 기록의 역할입니다.
manifest_path = ROOT / "data/splits/physionet-2012/revisions/v2/split-manifest.csv"
if not manifest_path.is_file():
    raise FileNotFoundError(
        f"분할 표가 없습니다: {manifest_path.as_posix()}. "
        "`uv run python scripts/setup_course.py --data-only`로 실습 데이터를 준비합니다."
    )

lineage = json.loads(lineage_path.read_text(encoding="utf-8"))
manifest = pd.read_csv(manifest_path)

# 역할별 행 수만 셉니다. test.csv 내용은 읽지 않습니다.
role_counts = manifest["role"].value_counts().sort_index()
role_counts

### 2. 데이터 상태와 파일 지문 계산

`uv run dvc status`는 지금 데이터가 선언과 어긋났는지를 보여 줍니다. 전역 `dvc`가 없어도 과정 환경에서 실행합니다. 상태가 더러워도 출력을 보고 다음 지문 계산으로 이어갑니다. 이어서 학습 파일과 검증 파일의 SHA-256 지문을 직접 계산해 선언과 비교합니다.

잠금 파일 지문은 출력만 합니다. 선언과 달라도 값을 고치지 말고 그대로 적습니다.

In [ ]:
import subprocess
from hashlib import sha256

def file_digest(path: Path) -> str:
    """파일 바이트의 SHA-256 지문을 돌려줍니다."""
    if not path.is_file():
        raise FileNotFoundError(
            f"파일이 없습니다: {path.as_posix()}. "
            "`uv run python scripts/setup_course.py --data-only`로 실습 데이터를 준비합니다."
        )
    return sha256(path.read_bytes()).hexdigest()

# 학생 VM은 전역 dvc 없이 uv로 실행합니다. 더러운 상태는 출력만 남깁니다.
try:
    status = subprocess.run(
        ["uv", "run", "dvc", "status"],
        cwd=ROOT,
        capture_output=True,
        text=True,
    )
except FileNotFoundError as error:
    raise FileNotFoundError(
        "`uv`를 찾을 수 없습니다. 과정 환경에서 노트북을 실행합니다."
    ) from error
if status.returncode != 0 and not status.stdout.strip():
    raise RuntimeError(
        status.stderr.strip() or "`uv run dvc status`를 시작할 수 없습니다."
    )
print(status.stdout.strip() or "변경이 감지되지 않았습니다.")

train_info = lineage["role_datasets"]["train"]
valid_info = lineage["role_datasets"]["valid"]
train_path = ROOT / train_info["path"]
valid_path = ROOT / valid_info["path"]

# 학습/검증 파일만 읽습니다. CSV를 표로 열지 않습니다.
train_digest = file_digest(train_path)
valid_digest = file_digest(valid_path)
lock_digest = file_digest(ROOT / "dvc.lock")
declared_lock = lineage["configuration"]["dvc_lock_sha256"]

pd.DataFrame(
    {
        "계산한 지문": [train_digest, valid_digest, lock_digest],
        "선언된 지문": [train_info["sha256"], valid_info["sha256"], declared_lock],
        "같은가": [
            train_digest == train_info["sha256"],
            valid_digest == valid_info["sha256"],
            lock_digest == declared_lock,
        ],
    },
    index=["train.csv", "valid.csv", "dvc.lock"],
)

## 해석과 기록

### 3. 같은 지문과 다른 지문을 구분해 적기

학습/검증 지문이 선언과 같으면, 지금 폴더의 개발용 데이터 내용이 현재 분할과 같다고 말할 수 있습니다. 이 확인은 데이터 신원만 답합니다. 자동 검증 성공이나 모델 승인을 대신하지 않습니다.

잠금 파일 지문은 다를 수 있습니다. 공식 기록은 잠금 스냅샷이 작업 폴더에 항상 복원돼 있다고 약속하지 않습니다. 다른 값을 맞추려고 고치지 말고, 계산한 값과 선언 값을 판단 기록의 데이터 품질 칸에 보조 확인으로만 붙입니다.

## 결과 점검

아래 검사는 역할 수와 개발용 파일 지문이 선언과 같은지 확인합니다.

In [ ]:
# 선언한 역할 수와 직접 센 수가 같아야 합니다.
assert lineage["revision"] == "v2"
assert int(role_counts["train"]) == 2900
assert int(role_counts["valid"]) == 600
assert int(role_counts["test"]) == 400
assert int(role_counts["operational"]) == 100
assert train_digest == train_info["sha256"]
assert valid_digest == valid_info["sha256"]
assert train_path.name == "train.csv"
assert valid_path.name == "valid.csv"
print(
    "학습/검증 지문을 선언과 대조했습니다. "
    "공식 평가용 test와 operational 파일은 열지 않았습니다."
)

## 다음 확인

본편으로 돌아가 데이터 품질 노트북과 자동 검증 해석을 닫습니다. 이 지문을 모델 승인 근거로 옮기지 않습니다. 시간이 더 있으면 2장의 선택 노트북에서 연습용 실험 기록을 남깁니다.